# Tumulus LiDAR Detector — live demo

Finds burial mounds (kurgans) in Romania's 0.5 m airborne LiDAR, in your browser.

1. **Runtime → Run all** — once (~1 min); the map appears below.
2. **Click a point** — start inside the **green demo area** (a point there is already picked for you).
3. Press **▶ Scan** (~1 min/km) — 🔴 red = probable mound · 🟠 orange = to verify.

Shape is not proof: only field survey confirms a tumulus.

> ⚠ ANCPI, Romania's national LiDAR server, is temporarily offline after a cyberattack. The **green demo
> area** works from a mirror; the rest of the blue coverage resumes when ANCPI is back.

How it decides: a morphometric fingerprint and a CNN, fused *"looks like OR has the geometry of"* a mound —
details in [STUDY.md](https://github.com/ObuObuHub/tumulus-lidar-detector/blob/main/STUDY.md).


In [ ]:
#@title Step 1 · Setup — runs by itself on "Run all" { display-mode: "form" }
import os
if not os.path.isdir('tumulus-lidar-detector') and os.path.basename(os.getcwd()) != 'tumulus-lidar-detector':
    !git clone -q https://github.com/ObuObuHub/tumulus-lidar-detector.git
if os.path.basename(os.getcwd()) != 'tumulus-lidar-detector':
    %cd -q tumulus-lidar-detector
!pip install -q pyproj
print('✔ Ready — the map is below.')


In [ ]:
#@title Step 2+3 · The map: click a point → press ▶ Scan { display-mode: "form" }
import os, sys, json, base64, subprocess
import IPython
from IPython.display import display, HTML
from google.colab import output as _colab_output
import pyproj
_tf_to3844 = pyproj.Transformer.from_crs('EPSG:4326', 'EPSG:3844', always_xy=True)
_tf_to4326 = pyproj.Transformer.from_crs('EPSG:3844', 'EPSG:4326', always_xy=True)
_cov_rects = json.load(open('assets/laki3_coverage.geojson'))            # exact, for the point check
_cov_outline = json.load(open('assets/laki3_coverage_outline.geojson'))  # dissolved, for display

def _inside(lon, lat):
    for f in _cov_rects['features']:
        ring = f['geometry']['coordinates'][0]
        c = False; n = len(ring); j = n - 1
        for i in range(n):
            xi, yi = ring[i]; xj, yj = ring[j]
            if ((yi > lat) != (yj > lat)) and (lon < (xj - xi) * (lat - yi) / (yj - yi) + xi):
                c = not c
            j = i
        if c:
            return True
    return False

def _scan(lat, lon, km=1):
    lat = float(lat); lon = float(lon); km = max(1, min(5, int(km)))
    if not _inside(lon, lat):
        return IPython.display.JSON({'ok': False, 'msg': 'Outside the blue 0.5 m coverage.'})
    for _f in ('/tmp/zone_dets.csv', 'review/zone_view.jpg', 'review/zone_board.jpg', 'detected_mounds.csv'):
        if os.path.exists(_f):
            os.remove(_f)
    subprocess.run([sys.executable, 'tools/scan_zone_v4.py', str(lon), str(lat), str(km)], check=False)
    if not os.path.exists('review/zone_view.jpg'):
        return IPython.display.JSON({'ok': False, 'msg': 'No LiDAR tiles here — ANCPI is temporarily offline; try the green demo area.'})
    import pandas as pd
    n = 0; rows = []; maybe = []
    if os.path.exists('/tmp/zone_dets.csv'):
        det = pd.read_csv('/tmp/zone_dets.csv')
        kept = det[det['keep'] == 1].sort_values('score', ascending=False)
        n = len(kept)
        for i, (_, r) in enumerate(kept.iterrows(), 1):
            rows.append({'id': i, 'lon': round(float(r.lon), 6), 'lat': round(float(r.lat), 6),
                         'score': round(float(r.score), 3),
                         'cnn': (round(float(r.cnn), 3) if pd.notna(r.cnn) else ''),
                         'mahal': round(float(r.mahal), 2)})
        brd = det[(det['keep'] == 0) & (det['score'] >= 0.60)].sort_values('score', ascending=False).head(15)
        for _, r in brd.iterrows():
            maybe.append({'lon': round(float(r.lon), 6), 'lat': round(float(r.lat), 6),
                          'score': round(float(r.score), 3),
                          'cnn': (round(float(r.cnn), 3) if pd.notna(r.cnn) else ''),
                          'mahal': round(float(r.mahal), 2)})
    img = base64.b64encode(open('review/zone_view.jpg', 'rb').read()).decode()
    est, nord = _tf_to3844.transform(lon, lat); _half = km * 500
    _e0 = int((est - _half) // 1000); _e1 = int((est + _half) // 1000)
    _n0 = int((nord - _half) // 1000); _n1 = int((nord + _half) // 1000)
    _c1 = _tf_to4326.transform(_e0 * 1000, _n0 * 1000); _c2 = _tf_to4326.transform((_e1 + 1) * 1000, (_n1 + 1) * 1000)
    bbox = [round(min(_c1[1], _c2[1]), 6), round(min(_c1[0], _c2[0]), 6),
            round(max(_c1[1], _c2[1]), 6), round(max(_c1[0], _c2[0]), 6)]
    return IPython.display.JSON({'ok': True, 'n': n, 'img': img, 'rows': rows, 'maybe': maybe, 'bbox': bbox})

_colab_output.register_callback('notebook.scan', _scan)
display(HTML(open('assets/demo_map.html').read().replace('__COVOUT__', json.dumps(_cov_outline))))
